# Game Attribute Matching to Sales and Eilers Time Series

This notebook links each game in the AI-generated attribute table (bronze) to its sales/lease record and its Eilers historical time-series performance record. It then adds time-series features and writes an enriched silver attribute table.

**Inputs**
* `development_042_silver_sandbox.demand_forecast.game_attr_ai_bronz`: game attributes
* `development_042_silver_sandbox.demand_forecast.sales_silver_comp`: sales and lease data
* `development_042_silver_sandbox.demand_forecast.eilers_group_historical_time_series`: Eilers time-series performance (class, supplier, performance)

**Output**
* `development_042_silver_sandbox.demand_forecast.game_attr_ai_silver`: one row per attribute record, with its match results and time-series features

**Steps**
1. **Config**: set the table paths. Define `fuzzy_match_overrides`, a manually reviewed map from attribute names to sales names, and `no_sales_match_keys`, a list of titles known to have no sales match.
2. **Load and preview** the three source tables.
3. **Normalize titles**: `clean_game_name` lowercases each name, removes "overview" suffixes and non-alphanumeric characters, and collapses whitespace. The result is used as a join key. Sales names fall back from `normalized_game_name` to `sales_normalized_title`, then `friendly_game_name`. The notebook reports exact-match coverage, unmatched titles, and key collisions.
4. **Fuzzy suggestions**: cross-joins unmatched attribute titles with sales titles and scores them with Levenshtein similarity. It keeps the top 3 candidates per title with similarity >= 0.78. These are for manual review, and approved choices go into `fuzzy_match_overrides`.
5. **Build match lookup**: for each attribute key, resolves the sales match (the fuzzy override first, then an exact match) and the Eilers time-series match. It records `sales_match_type`, `has_sales_match`, and `has_eilers_ts_match`, and builds Python dictionaries of the mappings.
6. **Add time-series features**: from the Eilers data, it takes the latest `game_classification`, `parent_own_status`, `own_status`, and `game_category`, plus the maximum `months_since_release` for each game, and joins them on.
7. **Write** the enriched result to the silver table (overwrite mode).

In [0]:
game_attr_path= 'development_042_silver_sandbox.demand_forecast.game_attr_ai_bronz'
ts_perf_path='development_042_silver_sandbox.demand_forecast.eilers_group_historical_time_series'# contains the class and, supplier, performance  the games in ts_perf are all found in game_attr_path
sales_path='development_042_silver_sandbox.demand_forecast.sales_silver_comp'# contains sales and lease of games
rental_path='development_042_silver_sandbox.demand_forecast.lease_units_bronz'
dest_attr_silver='development_042_silver_sandbox.demand_forecast.game_attr_ai_silver'

In [0]:
fuzzy_match_overrides = {
    "25 50 100 play poker": "25 50 100 playpoker ac",
    "27 ways to win dazzling dollar": "27 ways to win dazzling dollars",
    "big horn bucks": "bighorn buck",
    "bonus pick poker": "bonus pick poker ac",
    "bonus sizzling 7s": "bonus sizzling 7",
    "cats wild serengeti": "cats wild serengeti prog",
    "cleopatra triple fortune link": "cleopatra triple fortune",
    "da vinci diamonds dual play": "davinci diamonds dual play",
    "double black pearl": "double black pearl 1l",
    "double da vinci diamonds": "double davinci diamonds",
    "dragon sphere": "dragonsphere",
    "egyptian link nefturis treasure": "egyptian link nefturis treasures",
    "egyptian link ramosis treasure": "egyptian link ramosis treasures",
    "five card plus 2 poker": "5 card plus 2 poker",
    "fortune link 4": "fortune link 4 1",
    "fu gui hao men 2 everlasting fortunes": "fu gui hao men ii everlasting fortunes",
    "gold and dragon ii": "gold and dragon 2",
    "golden tiger eternal dragon": "golden tigereternal dragon",
    "heart of the amazon": "heart of the amazon 21",
    "high on a hog": "high on the hog",
    "hot hit sizzling 7s free games": "hot hits sizzling 7s free games",
    "hot hit sizzling 7s": "hot roll sizzling 7s",
    "hot roll cash cash cash": "hot roll cah cah cah",
    "hot roll golden dice pink diamond": "hot roll golden dice pink diamond vid",
    "hot roll golden dice triple red hot 7s": "hot roll golden dice triple red hot 7s vid",
    "hot roll golden dice wild red 7s": "hot roll golden dice wild red sevens",
    "hot roll sizzling 7": "hot roll sizzling 7s",
    "hot roulette triple double diamonds": "hot roulette triple double diamond",
    "legend of the 3x2x phoenix wheel": "legend of the 3x2x phoenix",
    "little green men return": "little green men jr",
    "lucky llamas": "llucky llamas",
    "lucky triple gold shamrock sheamus": "lucky triple goldshamrock sheamus",
    "lucky triple gold smokeys den": "lucky triple goldsmokeys den",
    "major multipliers poker": "major multipliers poker ac",
    "mystery of the congo": "mystery of the congo 100l",
    "nuttin to it": "nuttn to it",
    "pick a multiplier poker": "pick a multiplier poker ac",
    "poppin multipliers poker": "slammin multipliers poker",
    "prince of lightning": "the prince of lightning",
    "spitfire triple red hot bells": "spitfire triple red hot 7s",
    "spitfire triple stars free game": "spitfire triple stars free games",
    "split n win link cats": "split n win linkcats",
    "state fair 2": "state fair",
    "stinkin rich skunks gone wild": "stinkin richskunks gone wild",
    "super happy fortune cat": "lbox super happy fortune cat",
    "super star poker ii": "super star poker action",
    "super times pay keno": "super times pay 5r9l",
    "super times pay super stacks": "super times pay super stacks action",
    "super wheelmania colossal cats": "super wheelmania link colossal cats",
    "super wheelmania legendary beauty": "super wheelmania link legendary beauty",
    "super wheelmania treasure fish": "super wheelmania link treasure fish",
    "super wheelmania vip riches": "super wheelmania link vip riches",
    "sure fire jackpots": "surefire jackpots",
    "the count of monte cristo": "count of monte cristo",
    "treasures of the pyramid": "treasures of the pyramids",
    "triple big wheel tales of the legends dairy": "triple big wheel tales of the legend dairy 10l",
    "triple diamond deluxe plus": "triple diamond deluxe",
    "triple double diamonds progressive free games": "triple double diamond progressive free games",
    "triple double gold bars": "triple double dollars",
    "triple double sizzling 7s": "triple double sizzling 7",
    "triple red hot 7s free games": "triple red hot 7 free games",
    "wild fury free games": "wild reef free games",
    "wild fury jackpot": "wild fury jackpots",
    "wild hits clover": "wild hits clovers",
    "wild wild wild poker": "wild wild wild poker ac",
}

no_sales_match_keys = [
    "3x2x wild diamonds",
    "double diamond 3x4x5x",
    "red hot respin double diamond",
    "wheel hottie",
]

In [0]:
game_attr_df = spark.table(game_attr_path)
ts_perf_df = spark.table(ts_perf_path)
sales_df = spark.table(sales_path)
rental_df=spark.table(rental_path)

In [0]:
display(game_attr_df)
display(ts_perf_df)
display(sales_df)
display(rental_df)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

def clean_game_name(col): # takes the overview thingy out
    name = F.lower(F.trim(col))
    name = F.regexp_replace(
        name,
        r"(?:\s+|[-_])overview(?:\s*[-_]\s*[^\s]+)?\s*$",
        ""
    )
    name = F.regexp_replace(name, r"[^a-z0-9]+", " ")
    return F.trim(F.regexp_replace(name, r"\s+", " "))

# Attribute file: clean game_name, ignoring the existing game_name_key.
attr_keyed = (
    game_attr_df
    .drop("game_name_key")
    .withColumn("join_game_name_key", clean_game_name(F.col("game_name")))
)

# Sales file: use normalized_game_name, with fallbacks when it is null.
sales_name = F.coalesce(
    F.col("normalized_game_name"),
    F.col("sales_normalized_title"),
    F.col("friendly_game_name")
)

sales_keyed = sales_df.withColumn(
    "join_game_name_key",
    clean_game_name(sales_name)
)

# One row per title for coverage and unmatched-title checks.
attr_titles = attr_keyed.groupBy("join_game_name_key").agg(
    F.min("game_name").alias("attribute_game_name")
)

sales_titles = sales_keyed.where(
    F.col("join_game_name_key").isNotNull() &
    (F.col("join_game_name_key") != "")
).groupBy("join_game_name_key").agg(
    F.min("normalized_game_name").alias("sales_normalized_name"),
    F.min("friendly_game_name").alias("example_friendly_name")
)

matched_titles = attr_titles.join(
    sales_titles.select("join_game_name_key"),
    on="join_game_name_key",
    how="inner"
)

unmatched_titles = attr_titles.join(
    sales_titles.select("join_game_name_key"),
    on="join_game_name_key",
    how="left_anti"
)

print(f"Attribute titles: {attr_titles.count()}")
print(f"Exact matches: {matched_titles.count()}")
print(f"Unmatched attribute titles: {unmatched_titles.count()}")

display(unmatched_titles.orderBy("attribute_game_name"))
attr_key_collisions = (
    attr_keyed
    .groupBy("join_game_name_key")
    .agg(
        F.count("*").alias("rows"),
        F.collect_set("game_name").alias("source_names")
    )
    .where(F.col("rows") > 1)
)

display(attr_key_collisions)

In [0]:
unmatched = unmatched_titles.alias("a")
sales_candidates = sales_titles.alias("s")

fuzzy_pairs = (
    unmatched
    .crossJoin(sales_candidates)
    .withColumn(
        "edit_distance",
        F.levenshtein(
            F.col("a.join_game_name_key"),
            F.col("s.join_game_name_key")
        )
    )
    .withColumn(
        "similarity",
        F.lit(1.0) -
        F.col("edit_distance") /
        F.greatest(
            F.length(F.col("a.join_game_name_key")),
            F.length(F.col("s.join_game_name_key"))
        )
    )
    .select(
        F.col("a.attribute_game_name"),
        F.col("a.join_game_name_key").alias("attribute_key"),
        F.col("s.sales_normalized_name"),
        F.col("s.example_friendly_name"),
        F.col("s.join_game_name_key").alias("sales_key"),
        F.col("similarity")
    )
)

rank_window = Window.partitionBy("attribute_key").orderBy(
    F.desc("similarity")
)

fuzzy_suggestions = (
    fuzzy_pairs
    .withColumn("candidate_rank", F.row_number().over(rank_window))
    .where((F.col("candidate_rank") <= 3) & (F.col("similarity") >= 0.78))
    .orderBy("attribute_game_name", F.desc("similarity"))
)

display(fuzzy_suggestions)

In [0]:
from pyspark.sql.types import StructType, StructField, StringType

# Key: attribute join key or attribute game name
# Value: selected sales key or sales game name
# Add the reviewed fuzzy choices here.
if "fuzzy_match_overrides" not in globals():
    fuzzy_match_overrides = {
        # "attribute game key": "selected sales key",
    }

# Resolve a sales key to the displayed sales title.
sales_name_expr = F.coalesce(
    F.col("normalized_game_name"),
    F.col("sales_normalized_title"),
    F.col("friendly_game_name")
)

sales_title_lookup = (
    sales_keyed
    .withColumn("_sales_match_name", sales_name_expr)
    .where(
        F.col("join_game_name_key").isNotNull()
        & (F.col("join_game_name_key") != "")
    )
    .groupBy("join_game_name_key")
    .agg(F.min("_sales_match_name").alias("_sales_match_name"))
    .withColumnRenamed("join_game_name_key", "sales_key")
)

# Convert the manual dictionary into a Spark DataFrame.
mapping_rows = [
    (str(attribute_name), str(sales_name))
    for attribute_name, sales_name in fuzzy_match_overrides.items()
    if sales_name is not None
    and str(sales_name).strip().lower() not in {"", "none", "null"}
]

mapping_schema = StructType([
    StructField("attribute_name", StringType(), True),
    StructField("mapped_sales_name", StringType(), True),
])

mapping_df = (
    spark.createDataFrame(mapping_rows, schema=mapping_schema)
    .withColumn("join_game_name_key", clean_game_name(F.col("attribute_name")))
    .withColumn("mapped_sales_key", clean_game_name(F.col("mapped_sales_name")))
)

duplicate_override_keys = (
    mapping_df.groupBy("join_game_name_key")
    .count()
    .where(F.col("count") > 1)
)

if duplicate_override_keys.limit(1).count() > 0:
    display(duplicate_override_keys)
    raise ValueError("Multiple fuzzy overrides resolve to the same attribute key.")

mapped_sales_lookup = (
    mapping_df.alias("m")
    .join(
        sales_title_lookup.alias("s"),
        F.col("m.mapped_sales_key") == F.col("s.sales_key"),
        "left"
    )
    .select(
        F.col("m.join_game_name_key"),
        F.col("s._sales_match_name").alias("_mapped_sales_name")
    )
)

exact_sales_lookup = sales_title_lookup.select(
    F.col("sales_key").alias("join_game_name_key"),
    F.col("_sales_match_name").alias("_exact_sales_name")
)

# Find the Eilers time-series game-name column.
ts_name_candidates = [
    "game_name", "gamename", "game_title", "title", "game"
]
ts_columns_by_lower = {column.lower(): column for column in ts_perf_df.columns}
ts_name_col = next(
    (ts_columns_by_lower[name] for name in ts_name_candidates
     if name in ts_columns_by_lower),
    None
)

if ts_name_col is None:
    raise ValueError(
        "Could not find the game-name column in ts_perf_df. "
        f"Available columns: {ts_perf_df.columns}"
    )

eilers_lookup = (
    ts_perf_df
    .withColumn("_eilers_name", F.col(ts_name_col).cast("string"))
    .withColumn("_eilers_key", clean_game_name(F.col("_eilers_name")))
    .where(F.col("_eilers_key").isNotNull() & (F.col("_eilers_key") != ""))
    .groupBy("_eilers_key")
    .agg(F.min("_eilers_name").alias("_eilers_name"))
)

# Rental names are already unified in theme_name_eilers.
# Collapse the monthly rows to one lookup record per normalized name so this
# join does not multiply the attribute rows. Preserve the actual source name.
if "theme_name_eilers" not in rental_df.columns:
    raise ValueError("rental_df must contain theme_name_eilers.")

rental_lookup = (
    rental_df
    .select(F.col("theme_name_eilers").cast("string").alias("_rental_name"))
    .withColumn("_rental_key", clean_game_name(F.col("_rental_name")))
    .where(
        F.col("_rental_key").isNotNull()
        & ~F.col("_rental_key").isin("", "none", "null", "nan", "n a")
    )
    .groupBy("_rental_key")
    .agg(F.min("_rental_name").alias("_rental_name"))
)

# Create one match record per cleaned attribute name.
attribute_match_lookup = (
    attr_titles
    .select("join_game_name_key")
    .join(mapped_sales_lookup, on="join_game_name_key", how="left")
    .join(exact_sales_lookup, on="join_game_name_key", how="left")
    .join(
        eilers_lookup,
        F.col("join_game_name_key") == F.col("_eilers_key"),
        "left"
    )
    .drop("_eilers_key")
    .join(
        rental_lookup,
        F.col("join_game_name_key") == F.col("_rental_key"),
        "left"
    )
    .drop("_rental_key")
    .withColumn(
        "sales_match_type",
        F.when(F.col("_mapped_sales_name").isNotNull(), F.lit("fuzzy mapping"))
         .when(F.col("_exact_sales_name").isNotNull(), F.lit("exact"))
         .otherwise(F.lit("none"))
    )
    .withColumn(
        "has_sales_match",
        F.col("_mapped_sales_name").isNotNull()
        | F.col("_exact_sales_name").isNotNull()
    )
    .withColumn(
        "to_sales_name_match",
        F.coalesce(
            F.col("_mapped_sales_name"),
            F.col("_exact_sales_name"),
            F.lit("None")
        )
    )
    .withColumn(
        "has_eilers_ts_match",
        F.col("_eilers_name").isNotNull()
    )
    .withColumn(
        "to_eilers_ts_name_match",
        F.coalesce(F.col("_eilers_name"), F.lit("None"))
    )
    .withColumn(
        "has_rental_match",
        F.col("_rental_name").isNotNull()
    )
    .withColumn(
        "to_rental_name_match",
        F.coalesce(F.col("_rental_name"), F.lit("None"))
    )
    .select(
        "join_game_name_key",
        "to_sales_name_match",
        "has_sales_match",
        "sales_match_type",
        "to_eilers_ts_name_match",
        "has_eilers_ts_match",
        "to_rental_name_match",
        "has_rental_match"
    )
)

# Keep every attribute row and attach its match results.
attribute_match_df = attr_keyed.join(
    attribute_match_lookup,
    on="join_game_name_key",
    how="left"
)

# Python dictionaries keyed by the cleaned attribute game name.
attribute_to_sales_name_mapping = {
    row["join_game_name_key"]: row["to_sales_name_match"]
    for row in attribute_match_lookup
        .select("join_game_name_key", "to_sales_name_match")
        .distinct()
        .collect()
}

attribute_to_eilers_ts_name_mapping = {
    row["join_game_name_key"]: row["to_eilers_ts_name_match"]
    for row in attribute_match_lookup
        .select("join_game_name_key", "to_eilers_ts_name_match")
        .distinct()
        .collect()
}

attribute_to_rental_name_mapping = {
    row["join_game_name_key"]: row["to_rental_name_match"]
    for row in attribute_match_lookup
        .select("join_game_name_key", "to_rental_name_match")
        .collect()
}

display(
    attribute_match_df.select(
        "game_name",
        "to_sales_name_match",
        "has_sales_match",
        "sales_match_type",
        "to_eilers_ts_name_match",
        "has_eilers_ts_match",
        "to_rental_name_match",
        "has_rental_match"
    )
)

display(
    attribute_match_df
    .where(
        ~F.col("has_sales_match")
        | ~F.col("has_eilers_ts_match")
        | ~F.col("has_rental_match")
    )
    .select(
        "game_name",
        "to_sales_name_match",
        "has_sales_match",
        "to_eilers_ts_name_match",
        "has_eilers_ts_match",
        "to_rental_name_match",
        "has_rental_match"
    )
)

In [0]:
# Use ts_games if it already exists; otherwise use the time-series DataFrame
# loaded earlier in the notebook.
ts_games_source = ts_games if "ts_games" in globals() else ts_perf_df

# Normalize names on both sides so punctuation/case differences don't block the join.
def ts_join_key(col):
    key = F.lower(F.trim(col))
    key = F.regexp_replace(key, r"\boverview(?:[- ]\d+)?\s*$", "")
    key = F.regexp_replace(key, r"[^a-z0-9]+", " ")
    return F.trim(key)

ts_prepared = (
    ts_games_source
    .withColumn("_ts_join_key", ts_join_key(F.col("game_name")))
    .withColumn("_ts_yearmonth", F.to_date(F.col("yearmonth")))
    .withColumn("_months_since_release_num", F.col("months_since_release").cast("double"))
)

# Keep the latest values for the categorical fields.
ts_latest_window = Window.partitionBy("_ts_join_key").orderBy(
    F.col("_ts_yearmonth").desc_nulls_last()
)

ts_latest = (
    ts_prepared
    .withColumn("_ts_row_num", F.row_number().over(ts_latest_window))
    .where(F.col("_ts_row_num") == 1)
    .select(
        "_ts_join_key",
        "game_classification",
        "parent_own_status",
        'parent_game',
        "own_status",
        "game_category",
        'game_release_date'
    )
)

# Keep the maximum months_since_release across the game's time-series rows.
ts_max_months = (
    ts_prepared
    .groupBy("_ts_join_key")
    .agg(F.max("_months_since_release_num").alias("max_months_since_release"))
)

ts_game_features = ts_latest.join(ts_max_months, on="_ts_join_key", how="left")

# Join the TS attributes onto the existing attribute-level output.
attribute_match_df = (
    attribute_match_df
    .withColumn("_ts_join_key", ts_join_key(F.col("to_eilers_ts_name_match")))
    .join(ts_game_features, on="_ts_join_key", how="left")
    .drop("_ts_join_key")
)

display(attribute_match_df)

In [0]:
attribute_match_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(dest_attr_silver)